# __Detailed Explanation of the UCF Crime Mini Dataset Code__

The Complete Code (split into different chunks of code) below is the first of those codes whichworked like a charm to train our model in the initial days. It is based on `ucf-crime-mini-dataset` and was originally used as a script in a `kaggle` notebook.

## __Initial Setup and Imports__

```python
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
```
This imports two important libraries:
- NumPy: Helps with mathematical operations and working with arrays (think of arrays as organized lists of numbers)
- Pandas: Helps with handling data tables and files like Excel spreadsheets

```python
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
```
_This code searches through folders in the '/kaggle/input' directory and prints all the filenames. It's helping the user see what data files are available._

```python
!pip install decord
```
_This installs a library called "decord" which is used for processing video files efficiently._

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import decord
from transformers import TimesformerForVideoClassification
import os
from glob import glob
```
Here the code imports several specialized libraries:
- Torch: A machine learning framework that helps build and train AI models
- Transforms: Tools to modify images (resize, normalize, etc.)
- DataLoader: Helps feed data into the model efficiently
- Decord: The video processing library just installed
- TimesformerForVideoClassification: A pre-made AI model specifically designed for classifying videos
- Glob: A tool to find files that match certain patterns

## __Setting Up the Dataset Path__

```python
DATASET_PATH = '/kaggle/input/ucfcrimeminidataset/dataset/'
```
_This tells the program where to find the dataset of crime-related videos._

## __Function to Extract Video Frames__

```python
def extract_frames(video_path, num_frames=8):
    vr = decord.VideoReader(video_path)
    frame_indices = torch.linspace(0, len(vr) - 1, num_frames).long().tolist()
    frames = [torch.tensor(vr[i].asnumpy()).permute(2, 0, 1) for i in frame_indices]
    return torch.stack(frames)
```
This function:
1. Opens a video file
2. Picks 8 frames evenly distributed throughout the video
3. Converts those frames into a format the AI can understand
4. Returns all frames stacked together as one item

_Think of this like taking 8 screenshots from different points in a movie to represent what the whole movie is about._

## __Image Transformations__

```python
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])
```
This creates a set of operations to apply to each video frame:
1. Resize all frames to 224×224 pixels (a standard size)
2. Convert the images to floating-point numbers
3. Normalize the colors so they have similar brightness and contrast

_This is like standardizing all the photos so they're easier to compare - like making sure all ID photos are the same size and lighting._

## __Loading the Video Dataset__

```python
def load_videos(root_dir):
    if not os.path.exists(root_dir):
        raise FileNotFoundError(f"Dataset path '{root_dir}' does not exist.")

    print("Loading videos from:", root_dir)
    print("Classes found:", os.listdir(root_dir))

    data, labels = [], []
    class_to_idx = {cls: idx for idx, cls in enumerate(sorted(os.listdir(root_dir)))}

    for cls in class_to_idx:
        class_dir = os.path.join(root_dir, cls)
        if os.path.isdir(class_dir):
            for video in glob(os.path.join(class_dir, '*.mp4')):
                frames = extract_frames(video)
                frames = transform(frames)
                data.append(frames)
                labels.append(class_to_idx[cls])

    return torch.stack(data), torch.tensor(labels)
```
This function:
1. Checks if the dataset folder exists
2. Prints what classes (types of crimes) are found in the dataset
3. Creates a mapping between crime types and numbers (e.g., "Robbery" = 0, "Assault" = 1)
4. For each video:
   - Extracts frames using the function defined earlier
   - Applies the transformations to standardize the frames
   - Stores the video data and its corresponding label
5. Returns all videos and their labels

## __Preparing Data for Training__

```python
train_data, train_labels = load_videos(DATASET_PATH)
test_data, test_labels = train_data, train_labels
train_dataset = TensorDataset(train_data, train_labels)
test_dataset = TensorDataset(test_data, test_labels)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
```
This section:
1. Loads all videos from the dataset folder
2. Uses the same data for both training and testing (not ideal but works for a small dataset)
3. Packages the data into formats that the AI model can use efficiently
4. Creates "loaders" that feed data to the model in small batches of 4 videos

_The "shuffle=True" means the training data is mixed up before use, which helps the AI learn better patterns instead of just memorizing the order._

## __Setting Up the AI Model__

```python
num_classes = len(os.listdir(DATASET_PATH))
model = TimesformerForVideoClassification.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()
```
This part:
1. Counts how many types of crimes are in the dataset
2. Loads a pre-trained video classification model called "TimesFormer" (created by Facebook/Meta)
3. Determines whether to use the graphics card (CUDA) or regular processor (CPU)
4. Moves the model to the appropriate hardware
5. Sets up the "optimizer" (AdamW) which controls how the model learns
6. Defines the "loss function" (CrossEntropyLoss) which measures how wrong the model's predictions are

_Think of this like bringing in an expert who already knows a lot about identifying objects in videos, and then preparing to teach them specifically about identifying crimes._

## __Training the Model__

```python
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for videos, labels in train_loader:
        videos, labels = videos.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(videos).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")
```
This is the training loop, which:
1. Sets up to train for 5 full passes (epochs) through all the data
2. For each pass:
   - Puts the model in training mode
   - For each batch of videos:
     - Moves the videos and labels to the right hardware
     - Resets the optimizer
     - Runs the videos through the model to get predictions
     - Calculates how wrong the predictions are (the loss)
     - Uses "backpropagation" to figure out how to improve the model
     - Updates the model's parameters to make it better
   - Prints the average loss for that pass

_This is like showing the expert many examples of crimes, telling them when they're wrong, and helping them improve their identification skills._

## __Evaluating the Model__

```python
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for videos, labels in test_loader:
        videos, labels = videos.to(device), labels.to(device)
        outputs = model(videos).logits
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')
```
Finally, this code:
1. Switches the model to evaluation mode
2. Disables gradient calculations to save memory during testing
3. For each batch of test videos:
   - Gets the model's predictions
   - Counts how many predictions match the actual labels
4. Calculates and prints the overall accuracy percentage

_This is like giving the expert a final exam to see how well they can identify crimes in videos they haven't seen before._